In [ ]:
# 1 - cfg (Kaggle)
import os, sys, json, shutil
from pathlib import Path

REPO = "/kaggle/working/ProtoFormer"
GIT_URL = "https://github.com/pratamabintang/PF.git"
BRANCH = "main"
DATA = Path("/kaggle/working/data")
SAM2_CKPT = "/kaggle/working/sam2_hiera_l.pth"
SAM2_HF_REPO = "facebook/sam2-hiera-large"
SAM2_HF_FILE = "sam2_hiera_large.pt"

HP = dict(
    benchmark="pascal", fold=0,
    bsz=2, lr=1e-3, niter=30, nworker=2, reduce_dim=64,
    amp="auto", patience=8, val_interval=1,
)

def train_args(shot):
    return ["train.py",
        "--datapath", str(DATA),
        "--benchmark", HP["benchmark"],
        "--fold", str(HP["fold"]),
        "--shot", str(shot),
        "--bsz", str(HP["bsz"]),
        "--lr", str(HP["lr"]),
        "--niter", str(HP["niter"]),
        "--nworker", str(HP["nworker"]),
        "--reduce_dim", str(HP["reduce_dim"]),
        "--amp", HP["amp"],
        "--sam2_checkpoint", SAM2_CKPT,
        "--patience", str(HP["patience"]),
        "--val_interval", str(HP["val_interval"]),
        "--logpath", f"pascal_fold{HP['fold']}_{shot}shot",
    ]

print(HP); print("REPO", REPO, "DATA", DATA)
import torch; print(torch.__version__, "cuda", torch.cuda.is_available())


In [ ]:
# 2 - clone repo
try:
    import git
except ImportError:
    import subprocess; subprocess.run([sys.executable, "-m", "pip", "install", "-q", "GitPython"], check=True)
    import git
if not Path(REPO).exists():
    git.Repo.clone_from(GIT_URL, REPO, branch=BRANCH)
else:
    import os; os.chdir(REPO)
    repo = git.Repo(REPO)
    repo.git.fetch("origin"); repo.git.checkout(BRANCH); repo.git.reset("--hard", f"origin/{BRANCH}")
os.chdir(REPO)
assert Path(REPO, "train.py").exists()
sys.path.insert(0, os.path.join(REPO, "model"))
import subprocess, importlib.util
for mod, pkg in (("hydra", "hydra-core"), ("omegaconf", "omegaconf")):
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
print("repo OK")


In [ ]:
# 3 - deps
import subprocess
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-3000:]
import torch, torchvision
print(torch.__version__, torchvision.__version__)
sys.path.insert(0, os.path.join(REPO, "model"))
import sam2; print("sam2 OK", sam2.__file__)
print("deps OK")


In [ ]:
# 4 - VOC2012 dari HuggingFace
from datasets import load_dataset
voc = DATA / "VOC2012"
for d in (voc / "JPEGImages", voc / "SegmentationClassAug", voc / "SegmentationClass"):
    d.mkdir(parents=True, exist_ok=True)
jpg_dir, seg_dir, img_dir = (voc / "JPEGImages", voc / "SegmentationClassAug", voc / "SegmentationClass")
if len(list(jpg_dir.glob("*.jpg"))) < 1000 or len(list(seg_dir.glob("*.png"))) < 1000:
    print("download VOC ...")
    ds = load_dataset("shijli/voc2012", "segmentation_aug")
    seen=set()
    for split_name in ("train","validation"):
        sp=ds[split_name]
        for i in range(len(sp)):
            row=sp[i]
            img_id=row["id"]
            if img_id in seen: continue
            seen.add(img_id)
            row["image"].convert("RGB").save(str(jpg_dir / f"{img_id}.jpg"), "JPEG", quality=95)
            row["mask"].save(str(seg_dir / f"{img_id}.png"), "PNG")
    ds2=load_dataset("shijli/voc2012", "segmentation")
    for split_name in ("train","validation"):
        sp=ds2[split_name]
        for i in range(len(sp)):
            row=sp[i]
            row["mask"].save(str(img_dir / f"{row['id']}.png"), "PNG")
    del ds, ds2
n_jpg=len(list(jpg_dir.glob("*.jpg"))); n_aug=len(list(seg_dir.glob("*.png"))); n_off=len(list(img_dir.glob("*.png")))
assert n_jpg >= 11831, f"JPEGImages kurang: {n_jpg}"
print("VOC OK", n_jpg, n_aug, n_off)


In [ ]:
# 5 - SAM2 checkpoint dari HuggingFace
import os
if Path(SAM2_CKPT).exists() and Path(SAM2_CKPT).stat().st_size > 100*1024*1024:
    print("SAM2 cached")
else:
    from huggingface_hub import hf_hub_download
    print("download SAM2 ... (~900 MB)")
    p = hf_hub_download(repo_id=SAM2_HF_REPO, filename=SAM2_HF_FILE, local_dir="/kaggle/working", local_dir_use_symlinks=False)
    shutil.copy(p, SAM2_CKPT)
assert Path(SAM2_CKPT).exists(), "SAM2 missing"
print("SAM2 OK", Path(SAM2_CKPT).stat().st_size // 1024 // 1024, "MB")


In [ ]:
# 6 - train 5-shot (satu-satunya training di Kaggle, kode terbaru dari git)
import runpy
sys.path.insert(0, os.path.join(REPO, "model"))
sys.argv = train_args(5)
runpy.run_path(f"{REPO}/train.py", run_name="__main__")


In [ ]:
# 7 - test 5-shot (inference val set, pakai weight best)
import runpy, glob
weights = sorted(glob.glob(f"{REPO}/**/best_*.pth", recursive=True))
print("testing with:", weights)
assert weights, "tidak ada weight tersimpan"
sys.path.insert(0, os.path.join(REPO, "model"))
sys.argv = ["test.py", "--datapath", str(DATA), "--benchmark", HP["benchmark"], "--fold", str(HP["fold"]), "--nshot", "5", "--load", weights[-1]]
runpy.run_path(f"{REPO}/test.py", run_name="__main__")


In [ ]:
# 8 - save: ke /kaggle/working + zip
import zipfile
best = list(Path(REPO).rglob("best_*.pth"))
print("weight:", [str(b) for b in best])
files=[]
for b in best:
    dst = Path("/kaggle/working") / b.name
    shutil.copy(b, dst); files.append(dst)
for p in sorted(Path(REPO).rglob("*.json")):
    if "hist" in p.name or "config" in p.name:
        shutil.copy(p, Path("/kaggle/working") / p.name); files.append(Path("/kaggle/working") / p.name)
for f in files: print(f.name, f.stat().st_size // 1024 // 1024, "MB")
assert files, "tidak ada weight tersimpan"
with zipfile.ZipFile("/kaggle/working/protoformer_out.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in files: z.write(f, f.name)
print("saved: /kaggle/working/protoformer_out.zip")
